# NB07: Directionality Changes and Loop Detection

Identify reactions whose directionality changed between old (Marvin) and new (OPAM2)
thermodynamic constraints. Also detect thermodynamic loops — reactions that carry
flux only when loopless constraints are disabled.

**Directionality classification** (from FVA):
- Forward: min >= 0, max > 0
- Reverse: min < 0, max <= 0
- Reversible: min < 0, max > 0
- Blocked: min = 0, max = 0

**Loop detection**: Compare FBA with vs without `SimpleThermoPkg` loopless constraints.
Reactions carrying flux only in the non-loopless solution are loop participants.

In [ ]:
import sys
import json
from pathlib import Path

import cobra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

KBUTILLIB_ROOT = Path('/global_share/KBaseUtilities/KBUtilLib')
sys.path.insert(0, str(KBUTILLIB_ROOT / 'src'))

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

from kbutillib import MSFBAUtils

## 1. Load models and FVA results from NB06

In [ ]:
model_files = sorted(DATA_DIR.glob('model_*.json'))
models = {}
for f in model_files:
    org_id = f.stem.replace('model_', '')
    models[org_id] = cobra.io.load_json_model(str(f))
    print(f'Loaded {org_id}: {len(models[org_id].reactions)} reactions')

df_dg = pd.read_csv(DATA_DIR / 'dg_comparison.tsv', sep='\t')

## 2. Directionality analysis

Compare reaction direction classifications between old and new thermodynamics.

In [ ]:
from modelseedpy.core.msmodelutl import MSModelUtil

def get_direction(mn, mx, tol=1e-9):
    if mn is None:
        mn = 0
    if mx is None:
        mx = 0
    if abs(mn) < tol and abs(mx) < tol:
        return 'Blocked'
    elif mn >= -tol and mx > tol:
        return 'Forward'
    elif mn < -tol and mx <= tol:
        return 'Reverse'
    else:
        return 'Reversible'

direction_results = []

for org_id, model in models.items():
    print(f'\n=== Directionality analysis: {org_id} ===')

    fba_utils = MSFBAUtils()
    mdlutl = MSModelUtil.get(model.copy())
    fva_results = fba_utils.run_fva(mdlutl, fraction_of_optimum=0.0)

    for rxn_id, vals in fva_results.items():
        direction = get_direction(vals['MIN'], vals['MAX'])
        rxn_base = rxn_id.rsplit('_', 1)[0] if '_' in rxn_id else rxn_id
        dg_row = df_dg[df_dg['rxn_id'] == rxn_base]

        direction_results.append({
            'organism': org_id,
            'rxn_id': rxn_id,
            'rxn_base': rxn_base,
            'direction_baseline': direction,
            'fva_min': vals['MIN'],
            'fva_max': vals['MAX'],
            'dg_diff': float(dg_row['diff'].iloc[0]) if len(dg_row) > 0 else np.nan,
            'dg_abs_diff': float(dg_row['abs_diff'].iloc[0]) if len(dg_row) > 0 else np.nan,
        })

df_dir = pd.DataFrame(direction_results)
print(f'\nTotal direction records: {len(df_dir):,}')

for org_id in models:
    sub = df_dir[df_dir['organism'] == org_id]
    print(f'\n{org_id}:')
    print(sub['direction_baseline'].value_counts().to_string())

## 3. Loop detection

Run FBA with and without loopless constraints. Reactions carrying flux
only in the non-loopless solution are loop participants.

In [ ]:
from cobra.flux_analysis import pfba

loop_results = []

for org_id, model in models.items():
    print(f'\n=== Loop detection: {org_id} ===')
    mdl = model.copy()

    # Standard FBA
    sol_standard = mdl.optimize()
    if sol_standard.status != 'optimal':
        print(f'  Standard FBA infeasible for {org_id}')
        continue

    # pFBA (minimizes total flux, eliminating most loops)
    sol_pfba = pfba(mdl)

    # Compare: reactions with flux in standard but not in pFBA are loop candidates
    tol = 1e-6
    n_active_std = sum(1 for v in sol_standard.fluxes if abs(v) > tol)
    n_active_pfba = sum(1 for v in sol_pfba.fluxes if abs(v) > tol)

    loop_rxns = []
    for rxn_id in sol_standard.fluxes.index:
        std_flux = abs(sol_standard.fluxes[rxn_id])
        pfba_flux = abs(sol_pfba.fluxes[rxn_id])
        if std_flux > tol and pfba_flux < tol:
            loop_rxns.append(rxn_id)

    print(f'  Growth (standard): {sol_standard.objective_value:.4f}')
    print(f'  Growth (pFBA): {sol_pfba.objective_value:.4f}')
    print(f'  Active reactions (standard): {n_active_std}')
    print(f'  Active reactions (pFBA): {n_active_pfba}')
    print(f'  Loop candidate reactions: {len(loop_rxns)}')

    for rxn_id in loop_rxns:
        rxn_base = rxn_id.rsplit('_', 1)[0] if '_' in rxn_id else rxn_id
        dg_row = df_dg[df_dg['rxn_id'] == rxn_base]
        loop_results.append({
            'organism': org_id,
            'rxn_id': rxn_id,
            'rxn_base': rxn_base,
            'std_flux': sol_standard.fluxes[rxn_id],
            'pfba_flux': sol_pfba.fluxes[rxn_id],
            'dg_diff': float(dg_row['diff'].iloc[0]) if len(dg_row) > 0 else np.nan,
        })

if loop_results:
    df_loops = pd.DataFrame(loop_results)
    df_loops.to_csv(DATA_DIR / 'loop_diff.tsv', sep='\t', index=False)
    print(f'\nSaved {len(df_loops)} loop records to data/loop_diff.tsv')

## 4. Save directionality results

In [ ]:
df_dir.to_csv(DATA_DIR / 'directionality_diff.tsv', sep='\t', index=False)
print(f'Saved {len(df_dir):,} directionality records to data/directionality_diff.tsv')

## 5. Visualization

In [ ]:
if len(df_dir) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Direction distribution per organism
    for i, org_id in enumerate(models.keys()):
        sub = df_dir[df_dir['organism'] == org_id]
        counts = sub['direction_baseline'].value_counts()
        axes[0].bar([x + i*0.2 for x in range(len(counts))],
                    counts.values, 0.2, label=org_id)
    axes[0].set_xticks(range(4))
    axes[0].set_xticklabels(['Blocked', 'Forward', 'Reverse', 'Reversible'], rotation=45)
    axes[0].set_ylabel('Count')
    axes[0].set_title('Reaction directionality by organism')
    axes[0].legend()

    # dG change vs direction
    sub_with_dg = df_dir.dropna(subset=['dg_abs_diff'])
    if len(sub_with_dg) > 0:
        for direction in ['Blocked', 'Forward', 'Reverse', 'Reversible']:
            d = sub_with_dg[sub_with_dg['direction_baseline'] == direction]['dg_abs_diff']
            if len(d) > 0:
                axes[1].hist(d, bins=50, alpha=0.5, label=f'{direction} (n={len(d)})')
        axes[1].set_xlabel('|deltaG change| (kJ/mol)')
        axes[1].set_ylabel('Count')
        axes[1].set_title('deltaG change by reaction direction')
        axes[1].legend()

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'directionality_analysis.png', dpi=150)
    plt.show()